In [15]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [ ]:
import json
import boto3

BUCKET = "wellcomecollection-vhs-calm-adapter"

latest_version_map = {}

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        parts = key.split("/")
        if len(parts) != 3:
            continue

        item_id, version_str, filename = parts
        version = int(version_str)

        current = latest_version_map.get(item_id)
        if current is None or version > current[0]:
            latest_version_map[item_id] = (version, key)

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _download_object(latest) -> tuple[str, dict]:
    response = s3.get_object(Bucket=BUCKET, Key=latest[1][1])
    return latest[0], response["Body"].read().decode("utf-8")


calm_raw_works = {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {
        executor.submit(_download_object, latest): latest
        for latest in latest_version_map.items()
    }
    for future in as_completed(futures):
        record_id, record = future.result()
        calm_raw_works[record_id] = record

        if len(calm_raw_works) % 10000 == 0:
            print(len(calm_raw_works))

# print(result)

10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000


In [18]:
import polars as pl
import pickle

CALM_raw_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/calm-raw-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(json.loads(v)) for v in data.values()],
    })
    df.write_parquet(path)


save_parquet_snapshot(calm_raw_works, CALM_raw_WORKS_PATH)

In [14]:
len(latest_version_map)

408085